In [1]:
#モジュールのインポート
import pandas as pd
import numpy as np
import re
import json
from IPython.display import JSON
import requests
from bs4 import BeautifulSoup
import unicodedata
from collections import Counter
from janome.tokenizer import Tokenizer
from janome.analyzer import Analyzer
from janome.charfilter import UnicodeNormalizeCharFilter
from janome.tokenfilter import POSKeepFilter, POSStopFilter, CompoundNounFilter, LowerCaseFilter

In [4]:
#弊学科のホームページのニュース一覧から「お知らせ」のみを抽出し、月ごとに件数を集計してcsvに出力する
with open("./in/gsis_topics.html",encoding="utf-8") as f:
    html=f.read()
soup=BeautifulSoup(html,"html.parser")
newsbox=soup.find("div",class_="news-box-inner")
newslist=newsbox.find_all("li")
result={m:0 for m in range(1,13)}
for i in newslist:
    if i.find("em").text=="お知らせ":
        span=i.find("span").text
        month=int(span.split(".")[1])
        result[month]+=1
df=pd.DataFrame(list(result.items()),columns=["月","件数"])
df.to_csv("./out/JB24S087_news_monthly.csv",index=False,encoding="utf-8")

In [ ]:
#弊学学長の令和6年度学位授与式、令和7年度入学宣誓式の式辞の名詞（一部を除く）のみの形態素解析し、名詞の数、決まった5つの単語の出現回数をcsvに出力する

#関数実装
def get_message(file_path,start_phrase,end_phrase):
    """
    HTMLファイルから指定範囲のテキストを抽出し、形態素解析した結果を返す。
    """
    if  not "tokenizer" in globals():
        tokenizer = Tokenizer()
    char_filters=[UnicodeNormalizeCharFilter()]
    token_filters=[POSKeepFilter(['名詞']), 
                 POSStopFilter(['名詞,非自立'])]
    analyzer=Analyzer(char_filters=char_filters,tokenizer=tokenizer,token_filters=token_filters)
    with open(file_path, encoding = "utf-8") as f:
        ceremony_html = f.read()
    soup=BeautifulSoup(ceremony_html, "html.parser")
    start_index = soup.text.find(start_phrase)
    end_index = soup.text.find(end_phrase)
    if start_index == -1 or end_index == -1:
        print(f"Warning: 指定されたフレーズが {file_path} 内で見つかりませんでした。")
        return Counter()
    word_list1 = []
    word_list2 = []
    stop_words = {"兵庫", "県立", "大学", "みなさん", "たち"}
    target_text = analyzer.analyze(soup.text[start_index:end_index+len(end_phrase)])
    for i in target_text:
        word_list1.append(i.base_form)
    for j in word_list1:
        if len(j) >= 2 and j not in stop_words:
            word_list2.append(j)
    return Counter(word_list2)

#関数を実行し、それぞれの結果をデータフレームに統合
counter_r6=get_message("./in/g-ceremony-r6.html", "感謝や友情という花言葉をもつミモザの花が春の訪れを告げる今日", "きっと大丈夫だよ。")
counter_r7=get_message("./in/ceremony-r7.html", "柔らかな春の光に包まれて、", "おめでとう。兵庫県立大学にようこそ。")
data = []
target_keywords = ["困難", "感謝", "多様", "地域", "AI"]
labels = [("令和六年度学位授与式", counter_r6), ("令和七年度入学式", counter_r7)]
for title, counter in labels:
    row = {
        "式辞": title,
        "名詞数": counter.total()
    }
    for kw in target_keywords:
        row[kw] = counter[kw]
    data.append(row)
df_message=pd.DataFrame(data)
df_message.to_csv("./out/ceremony_message.csv",index=False,encoding="utf-8")

In [ ]:
#ホットペッパーグルメ　グルメサーチAPIから得られた「三宮」と「祇園」でのそれぞれ「寿司」というキーワードに関連する飲食店のキャッチコピーを形態素解析し、平均予算と名詞の出現回数上位4位をcsv出力

#ファイルパスと場所を与えると平均予算と上位4つの名詞の辞書を返す関数を実装
def analyze_shops(file_path,area_name):
    with open(file_path,encoding = "utf-8") as f:
        json_text = json.load(f)
    count = 0
    total_budget = 0
    word_list = []
    if  not "tokenizer" in globals():
        tokenizer = Tokenizer()
    char_filters = [UnicodeNormalizeCharFilter()]
    token_filters = [POSKeepFilter(['名詞',"一般"])]
    analyzer = Analyzer(char_filters = char_filters, tokenizer = tokenizer, token_filters = token_filters)
    for i in json_text["results"]["shop"]:
        count+=1
        match=re.search(r'～\s*(\d+)\s*円',i["budget"]["name"])
        if match:
            price=int(match.group(1))
        else:
            price=0
        total_budget+=price
        for j in analyzer.analyze(i["catch"]):
            if len(j.base_form) >= 2:
                word_list.append(j.base_form)
    average=np.round(total_budget/count,0)
    counter=Counter(word_list)
    counter_top=counter.most_common(4)
    result={
        "場所":area_name,
        "平均予算":average,
        "1位":counter_top[0][0] if len(counter_top) > 0 else "",
        "2位":counter_top[1][0] if len(counter_top) > 1 else "",
        "3位":counter_top[2][0] if len(counter_top) > 2 else "",
        "4位":counter_top[3][0] if len(counter_top) > 3 else ""
    }
    return result

#関数を実行し、三宮と祇園の情報をデータフレームに統合、csvに出力
results=[]
results.append(analyze_shops("./in/sannomiya.json","三ノ宮"))
results.append(analyze_shops("./in/gion.json","祇園"))
df=pd.DataFrame(results)
df.to_csv("./out/sumarry_shops.csv",index=False,encoding="utf-8")